# Task 5: Model training under leave-one-province-out CV
Rules enforced in every fold:
- Preprocessing is re-fit on fold-train rows only (nested, no leakage)
- Augmentation is generated from fold-train rows only
- Held-out province rows are real, untouched, and used only for scoring
- All 4 models see the identical synthetic rows within a fold (fair comparison)
Models: Linear (reported baseline), Ridge, Random Forest, constrained LightGBM.

In [1]:
# setup, data, and model roaster
import sys, json, time
from pathlib import Path
import numpy as np, pandas as pd, joblib

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.utils.augmentation import stratified_mixup

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

df = pd.read_csv(PROJECT_ROOT / "data/processed/merged_rice_yield_dataset.csv")
df["temp_mean_c"] = (df["temp_max_c"] + df["temp_min_c"]) / 2
df["temp_range_c"] = df["temp_max_c"] - df["temp_min_c"]

split_info = json.load(open(PROJECT_ROOT / "artifacts/split_provinces.json"))
df_train = df[df["province"].isin(split_info["train_provinces"])].copy()

X_train_raw = df_train.drop(columns=["yield_mt_ha", "province"])
y_train = df_train["yield_mt_ha"].to_numpy()
strata = (df_train["season_period"] + "|" + df_train["ecosystem"]
          + "|" + df_train["island_group"]).to_numpy()
province_of_row = df_train["province"].to_numpy()
provinces = sorted(df_train["province"].unique())

CAT_COLS = ["season_period", "ecosystem", "island_group"]
NUM_STD = ["n_application_rate", "p_application_rate", "k_application_rate",
           "total_fertilizer_kg_ha", "fertilizer_diversification_index",
           "rainfall_mm", "solar_radiation_mj_m2", "soil_ph",
           "temp_mean_c", "temp_range_c"]
NUM_LOG = ["area_harvested_ha"]

def make_preprocessor():
    return ColumnTransformer(transformers=[
        ("num_std", Pipeline([("scaler", StandardScaler())]), NUM_STD),
        ("num_log", Pipeline([("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
                              ("scaler", StandardScaler())]), NUM_LOG),
        ("cat", Pipeline([("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))]), CAT_COLS),
    ])

def make_models():
    return {
        "linear_baseline": LinearRegression(),
        "ridge": Ridge(alpha=1.0, random_state=42),
        "random_forest": RandomForestRegressor(
            n_estimators=500, min_samples_leaf=2, random_state=42, n_jobs=-1),
        "lightgbm": LGBMRegressor(
            n_estimators=200, learning_rate=0.05, num_leaves=7, max_depth=3,
            min_child_samples=25, colsample_bytree=0.8,
            reg_alpha=1.0, reg_lambda=5.0, random_state=42, verbose=-1),
    }

print(f"Training provinces: {len(provinces)} | rows: {len(df_train)}")

Training provinces: 60 | rows: 229


In [2]:
# LOPO cv engine
def lopo_cv(multiplier, province_list, base_seed=42):
    model_names = list(make_models().keys())
    oof = {name: np.full(len(df_train), np.nan) for name in model_names}
    fold_rmse = {name: [] for name in model_names}
    fold_mae = {name: [] for name in model_names}

    for f, prov in enumerate(province_list):
        tr = province_of_row != prov
        va = province_of_row == prov

        prep = make_preprocessor()
        Xf = prep.fit_transform(X_train_raw[tr])
        Xv = prep.transform(X_train_raw[va])
        if hasattr(Xf, "toarray"):
            Xf, Xv = Xf.toarray(), Xv.toarray()
        yf, yv, sf = y_train[tr], y_train[va], strata[tr]

        Xa, ya, wa = stratified_mixup(Xf, yf, sf, multiplier=multiplier,
                                      random_state=base_seed + f)

        for name, model in make_models().items():
            model.fit(Xa, ya, sample_weight=wa)
            pred = model.predict(Xv)
            oof[name][va] = pred
            fold_rmse[name].append(np.sqrt(mean_squared_error(yv, pred)))
            fold_mae[name].append(mean_absolute_error(yv, pred))

    return oof, fold_rmse, fold_mae

In [3]:
# metrics summary
def summarize(oof, fold_rmse, fold_mae, multiplier):
    rows = []
    for name in oof:
        mask = ~np.isnan(oof[name])
        yt, yp = y_train[mask], oof[name][mask]
        rows.append({
            "model": name, "multiplier": multiplier,
            "oof_rmse": np.sqrt(mean_squared_error(yt, yp)),
            "oof_mae": mean_absolute_error(yt, yp),
            "oof_r2": r2_score(yt, yp),
            "fold_rmse_mean": np.mean(fold_rmse[name]),
            "fold_rmse_std": np.std(fold_rmse[name]),
            "fold_mae_mean": np.mean(fold_mae[name]),
            "fold_mae_std": np.std(fold_mae[name]),
            "n_folds": len(fold_rmse[name]),
        })
    return pd.DataFrame(rows)

In [4]:
# smoke test
SMOKE = True
mults = [1] if SMOKE else [1, 2, 3, 5]
prov_list = provinces[:5] if SMOKE else provinces

results, oof_store = [], {}
t0 = time.time()
for m in mults:
    oof, frmse, fmae = lopo_cv(m, prov_list)
    results.append(summarize(oof, frmse, fmae, m))
    for name in oof:
        oof_store[f"{name}_m{m}"] = oof[name]
print(f"Elapsed: {time.time() - t0:.1f}s")

smoke_df = pd.concat(results)
print(smoke_df.round(4).to_string(index=False))
print("\nAny NaN in OOF predictions:", any(np.isnan(v).any() for v in oof_store.values()))

Elapsed: 3.3s
          model  multiplier  oof_rmse  oof_mae  oof_r2  fold_rmse_mean  fold_rmse_std  fold_mae_mean  fold_mae_std  n_folds
linear_baseline           1    0.6076   0.4758 -0.3386          0.5336         0.2907         0.4758        0.2685        5
          ridge           1    0.6070   0.4732 -0.3359          0.5315         0.2933         0.4732        0.2711        5
  random_forest           1    0.5920   0.4789 -0.2706          0.5386         0.2457         0.4789        0.2435        5
       lightgbm           1    0.6525   0.5112 -0.5434          0.5674         0.3221         0.5112        0.3120        5

Any NaN in OOF predictions: True
